In [2]:
import os
import glob
import pandas as pd
from pmdarima import auto_arima
import warnings

warnings.filterwarnings("ignore")

# 1. LOAD & PREPROCESS DATA
path = r"C:\Users\hp\Downloads\TimeSeries"
csv_files = glob.glob(os.path.join(path, "*.csv"))
df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

df = df.rename(columns={
    "prices_(£)": "price_pounds",
    "prices_unit_(£)": "price_unit_pounds",
    "names": "product_names"
})
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
df = df[~((df['supermarket'] == 'Morrisons') & (df['date'] <= '2024-01-27'))]
df = df.dropna(subset=['price_pounds', 'date', 'own_brand', 'category'])
df = df[df['price_pounds'] > 0]
df = df.drop_duplicates(subset=['supermarket', 'product_names', 'date'])

supermarkets = df['supermarket'].unique()

# =====================================================================
# IDE 1: FORECASTING OWN BRAND VS NATIONAL BRAND (Per Supermarket)
# =====================================================================
brand_types = df['own_brand'].unique()
df_forecast_ide1 = pd.DataFrame()

for market in supermarkets:
    for brand in brand_types:
        df_filtered = df[(df['supermarket'] == market) & (df['own_brand'] == brand)]
        if df_filtered.empty or len(df_filtered['date'].unique()) < 14:
            continue
            
        daily_price = df_filtered.groupby('date')['price_pounds'].mean().asfreq('D').ffill()
        train_size = int(len(daily_price) * 0.8)
        train_data, test_data = daily_price[:train_size], daily_price[train_size:]
        
        model = auto_arima(train_data, seasonal=True, m=7, trace=False, error_action='ignore', suppress_warnings=True)
        forecast, conf_int = model.predict(n_periods=len(test_data), return_conf_int=True, alpha=0.05)
        
        temp_df = pd.DataFrame({
            'date': test_data.index,
            'supermarket': market,
            'brand_type': f"Own Brand: {brand}",
            'actual_prices': test_data.values,
            'forecasted_prices': forecast,
            'min_expected_price': conf_int[:, 0],
            'max_expected_price': conf_int[:, 1]
        })
        df_forecast_ide1 = pd.concat([df_forecast_ide1, temp_df], ignore_index=True)

df_forecast_ide1.to_csv("Forecast_Ide1_OwnBrand.csv", index=False)


# =====================================================================
# IDE 2: FORECASTING TOP 3 KATEGORI TERATAS (Per Supermarket)
# =====================================================================
top_3_categories = df['category'].value_counts().head(3).index.tolist()
df_forecast_ide2 = pd.DataFrame()

for market in supermarkets:
    for cat in top_3_categories:
        df_cat = df[(df['supermarket'] == market) & (df['category'] == cat)]
        if df_cat.empty or len(df_cat['date'].unique()) < 14:
            continue
            
        daily_price_cat = df_cat.groupby('date')['price_pounds'].mean().asfreq('D').ffill()
        train_size = int(len(daily_price_cat) * 0.8)
        train_data, test_data = daily_price_cat[:train_size], daily_price_cat[train_size:]
        
        model = auto_arima(train_data, seasonal=True, m=7, trace=False, error_action='ignore', suppress_warnings=True)
        forecast, conf_int = model.predict(n_periods=len(test_data), return_conf_int=True, alpha=0.05)
        
        temp_df = pd.DataFrame({
            'date': test_data.index,
            'supermarket': market,
            'category': cat,
            'actual_prices': test_data.values,
            'forecasted_prices': forecast,
            'min_expected_price': conf_int[:, 0],
            'max_expected_price': conf_int[:, 1]
        })
        df_forecast_ide2 = pd.concat([df_forecast_ide2, temp_df], ignore_index=True)

df_forecast_ide2.to_csv("Forecast_Ide2_TopCategories.csv", index=False)